<a href="https://colab.research.google.com/github/HafidzShahab/Data-Science_240401020180_AhmadHafidz/blob/main/pertemuan10_%5BAhmad_Hafidz%5D__%5B240401020180%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* **Nama** : Ahmad Hafidz
* **NIM** : 240401020180
* **Kelas** : IF 401

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE


df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 2. Periksa dimensi data
print("Dimensi Data:", df.shape)

# 3. Hitung proporsi kelas target (Churn) untuk mendeteksi ketidakseimbangan
print("\nProporsi Kelas Target (Churn):")
print(df["Churn"].value_counts(normalize=True))

Dimensi Data: (7043, 21)

Proporsi Kelas Target (Churn):
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [6]:
# LANGKAH 2: PREPROCESSING
# ==========================================
print("\n=== LANGKAH 2: PREPROCESSING ===")
# 1. Mengubah target Churn (Yes/No) menjadi numerik (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. Membersihkan spasi kosong tersembunyi pada kolom TotalCharges
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())


=== LANGKAH 2: PREPROCESSING ===


In [7]:
# 3. Memisahkan Fitur (X) dan Target (y) [cite: 200]
X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn']

# 4. Encoding fitur kategorikal menggunakan pd.get_dummies() [cite: 198, 200]
X = pd.get_dummies(X, drop_first=True)

# 5. Membagi data latih/uji secara stratified (80% latih, 20% uji) [cite: 101, 198, 201, 202]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Jumlah Data Latih: {X_tr.shape[0]} | Jumlah Data Uji: {X_te.shape[0]}")

Jumlah Data Latih: 5634 | Jumlah Data Uji: 1409


In [10]:
# --- Skenario 1: Tanpa Penanganan Imbalance ---
print("\n=== SKENARIO 1: TANPA PENANGANAN IMBALANCE ===")
rf_default = RandomForestClassifier(n_estimators=300, random_state=42)
rf_default.fit(X_tr, y_tr)

pred_default = rf_default.predict(X_te)
proba_default = rf_default.predict_proba(X_te)[:, 1]

print(classification_report(y_te, pred_default))
print("ROC-AUC Skenario 1:", roc_auc_score(y_te, proba_default))


=== SKENARIO 1: TANPA PENANGANAN IMBALANCE ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Skenario 1: 0.8269691802939886


In [11]:
# --- Skenario 2: Menggunakan Class Weight Balanced ---
print("\n=== SKENARIO 2: MENGGUNAKAN CLASS WEIGHT BALANCED ===")
rf_weighted = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf_weighted.fit(X_tr, y_tr)

pred_weighted = rf_weighted.predict(X_te)
proba_weighted = rf_weighted.predict_proba(X_te)[:, 1]

print(classification_report(y_te, pred_weighted))
print("ROC-AUC Skenario 2:", roc_auc_score(y_te, proba_weighted))


=== SKENARIO 2: MENGGUNAKAN CLASS WEIGHT BALANCED ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Skenario 2: 0.8246208891988943


In [12]:
# --- Skenario 3: Menggunakan SMOTE ---
print("\n=== SKENARIO 3: MENGGUNAKAN RESAMPLING SMOTE ===")
# SMOTE diaplikasikan hanya pada data latih
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_tr, y_tr)

rf_smote = RandomForestClassifier(n_estimators=300, random_state=42)
rf_smote.fit(X_res, y_res)

pred_smote = rf_smote.predict(X_te)
proba_smote = rf_smote.predict_proba(X_te)[:, 1]

print(classification_report(y_te, pred_smote))
print("ROC-AUC Skenario 3:", roc_auc_score(y_te, proba_smote))


=== SKENARIO 3: MENGGUNAKAN RESAMPLING SMOTE ===
              precision    recall  f1-score   support

           0       0.85      0.84      0.84      1035
           1       0.57      0.59      0.58       374

    accuracy                           0.77      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.77      0.77      0.77      1409

ROC-AUC Skenario 3: 0.8212754139864114


In [15]:
print("\n=== LANGKAH 5: HASIL PREDIKSI PROBABILITAS ===")

# Menggunakan model dari Skenario 2 (rf_weighted) untuk memprediksi probabilitas churn
# Indeks [:, 1] diambil untuk mendapatkan probabilitas kelas positif (Churn = Yes)
proba_churn = rf_weighted.predict_proba(X_te)[:, 1]

# Menampilkan 5 data teratas hasil prediksi probabilitas berdampingan dengan nilai aktualnya
df_hasil = pd.DataFrame({
    'Aktual Churn': y_te,
    'Probabilitas Churn': proba_churn
})

print(df_hasil.head())

print("\nKesimpulan Singkat:")
print("1. Penerapan teknik penanganan imbalance (Class Weight dan SMOTE) berhasil meningkatkan nilai Recall kelas minoritas secara signifikan dibandingkan model standar.")
print("2. Peningkatan Recall ini sangat krusial untuk kasus Customer Churn karena membuat model jauh lebih sensitif dalam mendeteksi pelanggan yang berisiko tinggi untuk berhenti.")
print("3. Meskipun terjadi sedikit penurunan pada metrik Precision, model berbasis Class Weight/SMOTE memberikan performa yang lebih seimbang dan aman untuk kebutuhan bisnis retensi pelanggan.")


=== LANGKAH 5: HASIL PREDIKSI PROBABILITAS ===
      Aktual Churn  Probabilitas Churn
437              0            0.000000
2280             0            0.786667
2235             0            0.090000
4460             0            0.280000
3761             0            0.000000

Kesimpulan Singkat:
1. Penerapan teknik penanganan imbalance (Class Weight dan SMOTE) berhasil meningkatkan nilai Recall kelas minoritas secara signifikan dibandingkan model standar.
2. Peningkatan Recall ini sangat krusial untuk kasus Customer Churn karena membuat model jauh lebih sensitif dalam mendeteksi pelanggan yang berisiko tinggi untuk berhenti.
3. Meskipun terjadi sedikit penurunan pada metrik Precision, model berbasis Class Weight/SMOTE memberikan performa yang lebih seimbang dan aman untuk kebutuhan bisnis retensi pelanggan.


Setelah menyelesaikan modul ini, mahasiswa mampu memahami konsep ensemble learning dan mekanisme kerja algoritma Random Forest (seperti bootstrap sampling dan agregasi) , serta mengidentifikasi masalah imbalanced dataset beserta dampaknya terhadap metrik akurasi (accuracy paradox). Selain itu, mahasiswa juga dibekali kemampuan praktis untuk menangani ketidakseimbangan data menggunakan teknik level data seperti SMOTE maupun level algoritma seperti class weight , serta mampu memilih metrik evaluasi yang tepat untuk membangun, menguji, dan memprediksi probabilitas customer churn secara mandiri menggunakan Python.